In [10]:
import json

with open("2_item_info.json", "r", encoding="utf-8") as file:
    data = json.load(file)

geojson_fc = {"type": "FeatureCollection", "features": []}

for item in data:
    feature = {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": json.dumps(data["bbox"])}, "product_type": {"provider": item["provider"]}}

    geojson_fc["features"].append(feature)   
    pass


with open ("selsel.geojson", "w", encoding="utf-8") as file:
    json.dump(geojson_fc, file, indent=4, ensure_ascii=False)



TypeError: string indices must be integers, not 'str'

In [ ]:
import json

with open("3_raw_polygons.json", "r", encoding="utf-8") as file:
    data = json.load(file)

geojson_fc = {"type": "FeatureCollection", "features": []}


for item in data:
    correct_cords = [[lon, lat] for lat, lon in item["coordinates_lat_lon"]]

    feature = {"type": "Feature", "geometry": {"type": "Polygon",
                                               "coordinates": [correct_cords]},
            "properties": {"zone_id": item["id"], "area": item["area_sqkm"], "product_type": "WATER_MASK"}

In [ ]:
import logging
import time
from fastapi import BackgroundTasks, FastAPI
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO)

app = FastAPI(
    title="Geo Processing Service",
    description="Сервис асинхронной обработки географических снимков",
)

class OrderPayload(BaseModel):
    order_id: str
    input_image_path: str

def update_order_status(
    order_id: str, status: str, error_message: str = None):

    payload = {"status": status}
    if error_message:
        payload["error"] = error_message
    logging.info(f"[API UPDATE] Заказ {order_id} -> Статус: {status}")


# --- 3. Фоновая задача (Worker Task) ---
def process_geo_data_task(order_id: str, input_image_path: str):
    """Фоновая логика: меняет статус на processing, выполняет имитацию долгих вычислений
    и при успехе переводит в delivered, а при ошибке — в failed.
    """
    try:
        # 1. Перевод в статус processing
        update_order_status(order_id, "processing")
        logging.info(f"Начало обработки файла: {input_image_path}")

        # Эмуляция долгой геообработки (GDAL / SNAP)
        time.sleep(2)

        # Проверка формата исходного файла
        if not input_image_path.endswith((".tif", ".SAFE")):
            raise ValueError("Неподдерживаемый формат исходного снимка")

        # 2. Успешное завершение -> перевод в status delivered
        logging.info("Генерация маски завершена. Файлы записаны в /app/results")
        update_order_status(order_id, "delivered")

    except Exception as e:
        # 3. Перехват ошибки -> перевод в status failed
        logging.error(f"Ошибка при обработке заказа {order_id}: {str(e)}")
        update_order_status(order_id, "failed", error_message=str(e))


# --- 4. FastAPI Эндпоинт ---
@app.post("/api/process-order", status_code=202)
async def create_order(
    payload: OrderPayload, background_tasks: BackgroundTasks
):
    """Принимает JSON с задачей, передает её в фоновый режим
    и мгновенно возвращает ответ 202 Accepted.
    """
    background_tasks.add_task(
        process_geo_data_task,
        order_id=payload.order_id,
        input_image_path=payload.input_image_path,
    )

    return {
        "status": "accepted",
        "order_id": payload.order_id,
        "message": "Запрос принят и отправлен на фоновую обработку",
    }


# --- Запуск сервера прямо из файла ---
if __name__ == "__main__":
    import nest_asyncio
    import uvicorn

    # Разрешаем вложенные асинхронные циклы
    nest_asyncio.apply()

    if __name__ == "__main__":
        uvicorn.run(app, host="127.0.0.1", port=8000)

In [ ]:
import json

with open("3_raw_polygons.json", "r", encoding="utf-8") as file:
    data = json.load(file)


geojson_result = {"type": "FeatureCollection", "features": []}

for item in data:
    correct_coords = [[lon, lat] for lat, lon in item["coordinates_lat_lon"]]

    feature = {"type": "feature",
               "geometry": {"type": "Polygon",
                            "coordinates": [correct_coords]
                            },
                "properties": {"item_id": item["id"],
                               "area_sqkm": float(item["area_sqkm"]),
                                "product_type": "WATER_MASK"
                               }
               }
    geojson_result["features"].append(feature)

with open ("correct_geojson.geojson", "w", encoding="utf-8") as file:
    json.dump(geojson_result, file, indent=4, ensure_ascii=False)